# 02 · Modelos iniciales: 3 técnicas de ML y 3 arquitecturas de DL con 4 escenarios de balanceo
Se entrenan 24 combinaciones (6 técnicas × 4 escenarios de balanceo) y se comparan en **validación (2021–2022)**.
La evaluación en prueba se hace en el notebook 03, después del refinamiento.

Funciones utilizadas: `src/datos.py`, `src/balanceo.py`, `src/modelos_ml.py`, `src/redes.py` y `src/evaluacion.py`.

In [ ]:
# Celda de preparación: funciona en Google Colab y en un entorno local (VS Code / Jupyter)
import sys, os
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("acif104-brotes-eta").exists():
        !git clone https://github.com/FranciscoLopez27/acif104-brotes-eta.git
    os.chdir("acif104-brotes-eta")
    !git pull -q
    !pip -q install imbalanced-learn shap
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import datos as D, balanceo as B, evaluacion as E, modelos_ml as ML
from src.config import CAT, NUM, BALANCEOS, SEMILLA, AZUL, NARANJO, GRIS, VERDE, guardar, guardar_tabla

# Datos, partición y codificación (ajustada solo con entrenamiento)
p = D.particionar(D.preparar(D.cargar()))
cod = D.Codificador().fit(p["tr"][CAT + NUM])
M_tr, M_va = cod.transform(p["tr"][CAT + NUM]), cod.transform(p["va"][CAT + NUM])
y_tr, y_va = p["tr"].y.values, p["va"].y.values
print(M_tr.shape, M_va.shape, f"| línea base AUC-PR validación: {y_va.mean():.3f}")

# Escenarios de balanceo (solo entrenamiento)
datos_bal = {b: B.balancear(M_tr, y_tr, b, cod) for b in BALANCEOS}
tabla_bal = pd.DataFrame({b: {"Brotes de entrenamiento": len(v[1]), "% específico": round(v[1].mean() * 100, 1)}
                          for b, v in datos_bal.items()}).T
guardar_tabla(tabla_bal, "02_escenarios_balanceo")
tabla_bal

## 1. Técnicas de aprendizaje automático
Búsqueda en grilla de hiperparámetros (sin balanceo, criterio AUC-PR en validación).

In [ ]:
mejores_params, busqueda = {}, []
for nombre, grilla in ML.GRILLAS.items():
    for params in grilla:
        m = ML.ajustar(ML.crear_modelo(nombre, params, cod), M_tr, y_tr)
        busqueda.append({"Modelo": nombre, "Parámetros": params,
                         "AUC-PR validación": round(E.average_precision_score(y_va, ML.probabilidad(m, M_va)), 4)})
    mejores_params[nombre] = max([b for b in busqueda if b["Modelo"] == nombre],
                                 key=lambda b: b["AUC-PR validación"])["Parámetros"]
guardar_tabla(pd.DataFrame(busqueda), "02_busqueda_hiperparametros_ml")
print(mejores_params)
pd.DataFrame(busqueda)

In [ ]:
resultados, probas_va = [], {}
for nombre in ML.MODELOS_ML:
    for bal in BALANCEOS:
        Mb, yb, w = datos_bal[bal]
        t0 = time.time()
        m = ML.ajustar(ML.crear_modelo(nombre, mejores_params[nombre], cod), Mb, yb, w)
        seg = time.time() - t0
        pr = ML.probabilidad(m, M_va)
        resultados.append({"Familia": "ML", "Modelo": nombre, "Balanceo": bal,
                           **E.metricas(y_va, pr, E.umbral_para_recall(y_va, pr)), "Entrenamiento (s)": seg})
        probas_va[(nombre, bal)] = pr
res_ml = pd.DataFrame(resultados)
res_ml.round(3)

## 2. Arquitecturas de aprendizaje profundo (PyTorch)
MLP, MLP con *embeddings* y Transformer tabular, con parada temprana según el AUC-PR de validación.
La ponderación de clases se aplica con `pos_weight` en la función de pérdida.

In [ ]:
from src import redes as R
print("Dispositivo:", R.DISPOSITIVO)
prep = R.PreparadorRed(cod).fit(M_tr)

resultados_dl, historiales = [], {}
for nombre in R.ARQUITECTURAS:
    for bal in BALANCEOS:
        Mb, yb, w = datos_bal[bal]
        t0 = time.time()
        modelo, ep, hist = R.entrenar_red(nombre, prep, Mb, yb, M_va, y_va, ponderar=(bal == "Ponderación de clases"))
        seg = time.time() - t0
        pr = R.predecir(modelo, *prep.tensores(M_va))
        resultados_dl.append({"Familia": "DL", "Modelo": nombre, "Balanceo": bal,
                              **E.metricas(y_va, pr, E.umbral_para_recall(y_va, pr)),
                              "Entrenamiento (s)": seg, "Épocas": ep, "Parámetros": R.contar_parametros(modelo)})
        probas_va[(nombre, bal)] = pr
        historiales[(nombre, bal)] = hist
        print(f"{nombre:22s} | {bal:22s} | AUC-PR {resultados_dl[-1]['AUC-PR']:.3f} | mejor época {ep} | {seg:.0f} s")
res_dl = pd.DataFrame(resultados_dl)
res_dl.round(3)

## 3. Comparación de las 24 combinaciones

In [ ]:
res = pd.concat([res_ml, res_dl], ignore_index=True)
guardar_tabla(res, "02_validacion_24_combinaciones")
orden = ML.MODELOS_ML + list(R.ARQUITECTURAS)
tabla_ap = res.pivot(index="Modelo", columns="Balanceo", values="AUC-PR").loc[orden, BALANCEOS]
guardar_tabla(tabla_ap, "02_auc_pr_por_balanceo")

fig, ax = plt.subplots(figsize=(10, 3.8))
ancho = 0.8 / len(BALANCEOS)
for i, (bal, color) in enumerate(zip(BALANCEOS, [GRIS, AZUL, NARANJO, VERDE])):
    ax.bar(np.arange(len(tabla_ap)) + i * ancho - 0.4 + ancho / 2, tabla_ap[bal], ancho, label=bal, color=color)
ax.axhline(y_va.mean(), color="black", ls=":", lw=1, label=f"Línea base ({y_va.mean():.2f})")
ax.set_xticks(range(len(tabla_ap)), tabla_ap.index, rotation=15)
ax.set_ylabel("AUC-PR en validación")
ax.set_ylim(0, 0.75)
ax.legend(ncol=5, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.13))
guardar(fig, "fig_comparacion_balanceo.png")
tabla_ap.round(3)

## 4. Efecto del balanceo
Promedio de las 6 técnicas por escenario, con el umbral para exhaustividad ≥ 0,70 y con el umbral por defecto (0,5).

In [ ]:
efecto = res.groupby("Balanceo")[["AUC-PR", "AUC-ROC", "Recall", "Precisión", "F1", "Brier"]].mean().loc[BALANCEOS]
uf = []
for (nombre, bal), pr in probas_va.items():
    pred = (pr >= 0.5).astype(int)
    uf.append({"Balanceo": bal, "Recall (umbral 0,5)": E.recall_score(y_va, pred),
               "Precisión (umbral 0,5)": E.precision_score(y_va, pred, zero_division=0)})
uf = pd.DataFrame(uf).groupby("Balanceo").mean().loc[BALANCEOS]
efecto_balanceo = pd.concat([efecto, uf], axis=1)
guardar_tabla(efecto_balanceo, "02_efecto_balanceo")
efecto_balanceo.round(3)

## 5. Convergencia de las redes (escenario sin balanceo)
Pérdida de entrenamiento y de validación por época. Si la de validación sube mientras la de entrenamiento baja, hay sobreajuste.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, nombre in zip(axes, R.ARQUITECTURAS):
    h = historiales[(nombre, "Sin balanceo")]
    ep = int(res_dl.loc[(res_dl.Modelo == nombre) & (res_dl.Balanceo == "Sin balanceo"), "Épocas"].iloc[0])
    ax.plot(range(1, len(h["perdida_tr"]) + 1), h["perdida_tr"], color=GRIS, label="Entrenamiento")
    ax.plot(range(1, len(h["perdida_va"]) + 1), h["perdida_va"], color=NARANJO, label="Validación")
    ax.axvline(ep, color="black", ls=":", lw=1, label="Época seleccionada")
    ax.set_title(nombre)
    ax.set_xlabel("Época")
axes[0].set_ylabel("Pérdida (entropía cruzada)")
axes[0].legend(frameon=False, fontsize=8)
guardar(fig, "fig_convergencia_redes.png")